# Data Cleaning — Focus Sessions Dataset

This notebook takes the raw google form responses
and transforms it into a cleaned dataset
ready for analysis and visualization.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load the raw data

In [5]:
import pandas as pd
import numpy as np
import re
import math

RAW_PATH = "/content/drive/MyDrive/form_responses.csv"
PROCESSED_PATH = "/content/drive/MyDrive/cleaned_data.csv"

In [6]:
df_raw = pd.read_csv(RAW_PATH, keep_default_na=False, na_values=[""])
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
df_raw.head(3)

Loaded 22 rows, 20 columns


,Timestamp,What date did this study/work session take place?,What time did the session start?,What time did the session end?,What type of task were you doing?,How interesting was the task?,How difficult was the task?,How long did you plan to work/study? Enter the number of minutes. Example: 120,How many minutes after starting the session did you experience your first distraction? Enter a whole number. If no distraction occurred enter 0,How many distractions occurred? (Whole Number),How focused were you during the session?,Where did you study/work?,How would you describe the noise level during this session?,How many other people were around you during this session?,How much interaction did you have with other people during the session?,How many hours did you sleep the previous night? Example: 7.5,How many minutes of phone screen time had you accumulated before this session? Example: 145,What caffeine did you consume before the session?,Which focus strategies did you use,How effective was this study/work session overall? (Rate 1-5)
0,9/22/2026 15:14:02,9/21/2026,5:30:00 AM,7:00:00 AM,Studying,3,7,120,30,3,7,Home,Very Quiet,1-2 people,None,240,Option 1,None,"Noise cancelling headphones, Timer",3- Moderately effective
1,9/22/2026 15:19:28,9/22/2026,7:00:00 AM,9:00:00 AM,Studying,9,8,120,40,2,10,Bus,Quiet,6-10 people,None,4 hours,2 hours,None,None,5- Extremely effective
2,9/22/2026 15:34:19,12/9/2000,1:00:00 PM,2:00:00 PM,Reading,8,1,60 minutes,20 minutes,3,6,Home,Moderate,Alone,Very little,300 minutes,240 minutes,Soda,None,2- Slightly effective


## 2. Rename columns

The original Google Form question text makes unwieldy column names. We
rename them to short, snake_case names.

In [7]:
RENAME_MAP = {
    "Timestamp": "Timestamp",
    "What date did this study/work session take place?": "Date",
    "What time did the session start?": "Start_Time",
    "What time did the session end?": "End_Time",
    "What type of task were you doing?": "Task_Type",
    "How interesting was the task?": "Task_Interest",
    "How difficult was the task?": "Task_Difficulty",
    "How long did you plan to work/study? Enter the number of minutes. Example: 120": "Planned_Duration",
    "How many minutes after starting the session did you experience your first distraction? Enter a whole number. If no distraction occurred enter 0 ": "Time_to_First_Distraction",
    "How many distractions occurred? (Whole Number)": "Num_Distractions",
    "How focused were you during the session?": "Focus_Rating",
    "Where did you study/work?": "Location",
    "How would you describe the noise level during this session?": "Noise_Level",
    "How many other people were around you during this session?": "People_Present",
    "How much interaction did you have with other people during the session?": "Interaction_Level",
    "How many hours did you sleep the previous night? Example: 7.5": "Sleep_Hours",
    "How many minutes of phone screen time had you accumulated before this session? Example: 145": "Phone_Screen_Time",
    "What caffeine did you consume before the session?": "Caffeine",
    "Which focus strategies did you use": "Focus_Strategies",
    "How effective was this study/work session overall? (Rate 1-5)": "Effectiveness",
}

df = df_raw.rename(columns=RENAME_MAP)
df.columns.tolist()

['Timestamp',
 'Date',
 'Start_Time',
 'End_Time',
 'Task_Type',
 'Task_Interest',
 'Task_Difficulty',
 'Planned_Duration',
 'Time_to_First_Distraction',
 'Num_Distractions',
 'Focus_Rating',
 'Location',
 'Noise_Level',
 'People_Present',
 'Interaction_Level',
 'Sleep_Hours',
 'Phone_Screen_Time',
 'Caffeine',
 'Focus_Strategies',
 'Effectiveness']

## 3. Dates and times

**Data quality issue found:** one response has `date = 12/9/2000`, which
is almost certainly a birthdate accidentally typed into the session-date
field — it doesn't match the pattern of any other response and the
auto-logged form `Timestamp` for that same row is a normal 2026 date.

Rather than guess-correct the typed date, we use the auto-generated
`Timestamp` as the authoritative session date — it's machine-logged at
submission time and not prone to typos the way a free-text field is.

Start/end times are then standardized to 24-hour `HH:MM:SS` format.

In [8]:
df["Date"] = pd.to_datetime(df["Timestamp"], errors="coerce").dt.date

_start_dt = pd.to_datetime(df["Start_Time"], format="%I:%M:%S %p", errors="coerce")
_end_dt = pd.to_datetime(df["End_Time"], format="%I:%M:%S %p", errors="coerce")

df["Start_Time"] = _start_dt.dt.strftime("%H:%M:%S")
df["End_Time"] = _end_dt.dt.strftime("%H:%M:%S")

df[["Date", "Start_Time", "End_Time"]].head(5)

,Date,Start_Time,End_Time
0,2026-09-22,05:30:00,07:00:00
1,2026-09-22,07:00:00,09:00:00
2,2026-09-22,13:00:00,14:00:00
3,2026-09-22,15:30:00,16:30:00
4,2026-09-22,09:00:00,11:00:00


## 4. Total session duration

Computed from start and end time. A few sessions have an end time earlier
than the start time (e.g. 4:20 PM → 4:58 AM) — these genuinely crossed
midnight, so we add 24 hours in that case rather than returning a
negative duration.

We also flag unusually long sessions (> 5 hours) for manual review. A
few of these (e.g. an 8:00 PM → 9:20 AM "session") look more like an
AM/PM entry mistake than a real 13-hour study session, but we don't
silently guess-correct them — we flag them so they can be checked
against the original form response by whoever submitted it.

In [9]:
_start_min = _start_dt.dt.hour * 60 + _start_dt.dt.minute
_end_min = _end_dt.dt.hour * 60 + _end_dt.dt.minute
_duration = _end_min - _start_min
_duration = _duration.where(_duration >= 0, _duration + 24 * 60)

df["Total_Duration_Minutes"] = _duration
df["Duration_Flag_Long"] = df["Total_Duration_Minutes"] > 300

n_flagged = df["Duration_Flag_Long"].sum()
print(f"Flagged {n_flagged} sessions with duration > 5 hours for review:")
df.loc[df["Duration_Flag_Long"], ["Date", "Start_Time", "End_Time", "Total_Duration_Minutes"]]

Flagged 4 sessions with duration > 5 hours for review:


,Date,Start_Time,End_Time,Total_Duration_Minutes
5,2026-09-22,16:20:00,04:58:00,758
7,2026-09-22,12:30:00,18:00:00,330
16,2026-09-27,20:00:00,09:20:00,800
20,2026-09-30,13:30:00,02:50:00,800


## 5. Parsing messy numeric free-text fields

Several fields were collected as free text and came back inconsistently
formatted — plain numbers, `"90min"`, `"3 hours"`, `"Around 30"`, and in
one case the clearly-invalid text `"Option 1"` (a stray value that isn't
a time at all).

We write one small parser that:
- extracts the first number in the text,
- checks the remaining words against a short allow-list of recognized
  units (`hour`, `hours`, `hr`, `min`, `mins`, `minute`, `minutes`) and
  hedge words (`around`, `about`, `approx`, `roughly`),
- and returns a missing value if any *other*, unrecognized word is
  present — this is what correctly catches `"Option 1"` as invalid
  rather than misreading it as "1 minute".

This same parser, with a different output unit, is reused for:
`planned_duration`, `time_to_first_distraction`, `sleep_hours` (output in
hours), and `phone_screen_time` (output in minutes).

In [10]:
TIME_UNIT_WORDS = {"hour", "hours", "hr", "hrs", "min", "mins", "minute", "minutes"}
HEDGE_WORDS = {"around", "about", "approx", "approximately", "roughly"}
ALLOWED_WORDS = TIME_UNIT_WORDS | HEDGE_WORDS

def extract_number_and_unit(val):
    """Return (number, is_hours) or (None, None) if the text isn't a
    usable numeric/time value (e.g. 'Option 1')."""
    if pd.isna(val):
        return None, None
    s = str(val).strip().lower()
    words = re.findall(r"[a-zA-Z]+", s)
    if not all(w in ALLOWED_WORDS for w in words):
        return None, None  # contains an unrecognized word -> garbage entry
    match = re.search(r"\d+\.?\d*", s)
    if not match:
        return None, None
    num = float(match.group())
    is_hours = bool({"hour", "hours", "hr", "hrs"} & set(words))
    return num, is_hours

def parse_minutes(val):
    """Parse free text into a number of minutes."""
    num, is_hours = extract_number_and_unit(val)
    if num is None:
        return np.nan
    return num * 60 if is_hours else num

def parse_hours(val):
    """Parse free text into a number of hours (used for sleep_hours,
    where a bare number > 16 is assumed to be minutes, not hours)."""
    num, is_hours = extract_number_and_unit(val)
    if num is None:
        return np.nan
    if is_hours:
        return round(num, 2)
    if num > 16:
        return round(num / 60, 2)
    return round(num, 2)

In [11]:
df["Planned_Duration"] = df["Planned_Duration"].apply(parse_minutes)
df["Time_to_First_Distraction"] = df["Time_to_First_Distraction"].apply(parse_minutes)
df["Sleep_Hours"] = df["Sleep_Hours"].apply(parse_hours)
df["Phone_Screen_Time"] = df["Phone_Screen_Time"].apply(parse_minutes)

df[["Planned_Duration", "Time_to_First_Distraction", "Sleep_Hours", "Phone_Screen_Time"]].head(10)

,Planned_Duration,Time_to_First_Distraction,Sleep_Hours,Phone_Screen_Time
0,120.0,30.0,4.0,NaN
1,120.0,40.0,4.0,120.0
2,60.0,20.0,5.0,240.0
3,90.0,30.0,7.0,180.0
4,120.0,45.0,8.0,30.0
5,60.0,5.0,7.0,200.0
6,60.0,30.0,4.0,248.0
7,300.0,0.0,8.0,180.0
8,20.0,3.0,6.0,8.0
9,120.0,15.0,NaN,NaN


## 6. Distraction count

`"How many distractions occurred?"` should be a whole number, but one
response reads `"20+"`. We strip non-numeric characters, and round any
decimal up to the next whole number (so partial/uncertain counts don't
get silently truncated down).

In [12]:
def parse_distraction_count(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    match = re.search(r"\d+\.?\d*", s)
    if not match:
        return np.nan
    return int(math.ceil(float(match.group())))

df["Num_Distractions"] = df["Num_Distractions"].apply(parse_distraction_count)
df[["Num_Distractions"]]

,Num_Distractions
0,3
1,2
2,3
3,3
4,3
5,3
6,3
7,0
8,20
9,5


## 7. Standardize single-select categorical fields

For `task_type`, `location`, `noise_level`, and `caffeine`, we map each
response to the exact set of options offered on the form. Anything that
doesn't match an option exactly (e.g. `"Bus"`, `"Campus"`, `"Light lunch"`)
gets bucketed into `"Other"`, per the form's own design.

**One exception:** `"Meetings"` (plural) and `"Meeting"` are collapsed
into a single `"Meeting"` category — these are clearly the same answer
option, duplicated by a typo in the form itself, not two different task
types.

In [13]:
def map_to_allowed(val, allowed_set, other_label="Other"):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    for opt in allowed_set:
        if v.lower() == opt.lower():
            return opt
    return other_label

TASK_TYPES = {"Studying", "Coding", "Reading", "Meeting", "Chores", "Writing",
              "Homework/Assignments", "Research", "Other"}

def map_task_type(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    if v.lower() == "meetings":
        v = "Meeting"
    return map_to_allowed(v, TASK_TYPES)

LOCATIONS = {"Home", "Library", "Cafe", "Private Booth", "Other"}
NOISE_LEVELS = {"Very Quiet", "Quiet", "Moderate", "Noisy", "Very Noisy", "Other"}
CAFFEINE = {"Coffee", "Tea", "Energy drink", "Soda", "Don't remember", "None", "Other"}

df["Task_Type"] = df["Task_Type"].apply(map_task_type)
df["Location"] = df["Location"].apply(lambda v: map_to_allowed(v, LOCATIONS))
df["Noise_Level"] = df["Noise_Level"].apply(lambda v: map_to_allowed(v, NOISE_LEVELS))
df["Caffeine"] = df["Caffeine"].apply(lambda v: map_to_allowed(v, CAFFEINE))

df[["Task_Type", "Location", "Noise_Level", "Caffeine"]]

,Task_Type,Location,Noise_Level,Caffeine
0,Studying,Home,Very Quiet,None
1,Studying,Other,Quiet,None
2,Reading,Home,Moderate,Soda
3,Homework/Assignments,Library,Quiet,Coffee
4,Studying,Home,Very Quiet,None
5,Studying,Library,Moderate,Coffee
6,Meeting,Other,Moderate,None
7,Studying,Library,Very Quiet,None
8,Writing,Home,Quiet,None
9,Reading,Home,Moderate,Tea


## 8. One-hot encode focus strategies (multi-select)

This question allows multiple answers in one cell (e.g.
`"Background Music, Noise cancelling headphones, Timer"`), so a single
category column won't work. We split each response on commas and create
one boolean column per form option. Any item that isn't one of the
listed options (e.g. `"Airpods"`) is counted under `strategy_other`,
consistent with how we handle unlisted answers elsewhere.

In [14]:
STRATEGIES = ["Phone in another room", "Focus app", "Background Music", "Noise cancelling headphones", "Timer", "None", "Other"]

def one_hot_strategies(val):
    cols = {f"strategy_{s.lower().replace(' ', '_')}": 0 for s in STRATEGIES}
    if pd.isna(val) or str(val).strip() == "":
        return cols
    for item in str(val).split(","):
        item = item.strip()
        matched = False
        for s in STRATEGIES:
            if item.lower() == s.lower():
                cols[f"strategy_{s.lower().replace(' ', '_')}"] = 1
                matched = True
                break
        if not matched:
            cols["strategy_other"] = 1
    return cols

_strategy_cols = df["Focus_Strategies"].apply(one_hot_strategies).apply(pd.Series)
df = pd.concat([df, _strategy_cols], axis=1)
df[["Focus_Strategies"] + list(_strategy_cols.columns)]

,Focus_Strategies,strategy_phone_in_another_room,strategy_focus_app,strategy_background_music,strategy_noise_cancelling_headphones,strategy_timer,strategy_none,strategy_other
0,"Noise cancelling headphones, Timer",0,0,0,1,1,0,0
1,None,0,0,0,0,0,1,0
2,None,0,0,0,0,0,1,0
3,"Phone in another room, Focus app",1,1,0,0,0,0,0
4,None,0,0,0,0,0,1,0
5,None,0,0,0,0,0,1,0
6,"Background Music, Noise cancelling headphones,...",0,0,1,1,1,0,0
7,None,0,0,0,0,0,1,0
8,None,0,0,0,0,0,1,0
9,None,0,0,0,0,0,1,0


## 9. Extract numeric effectiveness rating

`effectiveness_raw` stores both a number and a label in one string (e.g.
`"3- Moderately effective"`). We pull out just the leading number as
`effectiveness` for analysis, and keep the original text for reference.

In [15]:
df["Effectiveness"] = df["Effectiveness"].str.extract(r"(\d+)").astype(float)
df[["Effectiveness"]]

,Effectiveness
0,3.0
1,5.0
2,2.0
3,4.0
4,3.0
5,3.0
6,4.0
7,5.0
8,2.0
9,2.0


## 10. Final column selection and data-quality summary

We assemble the cleaned columns into a tidy table, print a short summary
of missing values per column (so the state of the data is visible, not
hidden), and save the result.

In [17]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
df.head(10)

Missing values per column:
Sleep_Hours          1
Phone_Screen_Time    3
dtype: int64


,Timestamp,Date,Start_Time,End_Time,Task_Type,Task_Interest,Task_Difficulty,Planned_Duration,Time_to_First_Distraction,Num_Distractions,...,Effectiveness,Total_Duration_Minutes,Duration_Flag_Long,strategy_phone_in_another_room,strategy_focus_app,strategy_background_music,strategy_noise_cancelling_headphones,strategy_timer,strategy_none,strategy_other
0,9/22/2026 15:14:02,2026-09-22,05:30:00,07:00:00,Studying,3,7,120.0,30.0,3,...,3.0,90,False,0,0,0,1,1,0,0
1,9/22/2026 15:19:28,2026-09-22,07:00:00,09:00:00,Studying,9,8,120.0,40.0,2,...,5.0,120,False,0,0,0,0,0,1,0
2,9/22/2026 15:34:19,2026-09-22,13:00:00,14:00:00,Reading,8,1,60.0,20.0,3,...,2.0,60,False,0,0,0,0,0,1,0
3,9/22/2026 15:35:58,2026-09-22,15:30:00,16:30:00,Homework/Assignments,10,10,90.0,30.0,3,...,4.0,60,False,1,1,0,0,0,0,0
4,9/22/2026 16:18:07,2026-09-22,09:00:00,11:00:00,Studying,7,8,120.0,45.0,3,...,3.0,120,False,0,0,0,0,0,1,0
5,9/22/2026 17:02:42,2026-09-22,16:20:00,04:58:00,Studying,6,4,60.0,5.0,3,...,3.0,758,True,0,0,0,0,0,1,0
6,9/22/2026 17:05:13,2026-09-22,16:00:00,17:00:00,Meeting,6,4,60.0,30.0,3,...,4.0,60,False,0,0,1,1,1,0,0
7,9/22/2026 18:38:42,2026-09-22,12:30:00,18:00:00,Studying,7,8,300.0,0.0,0,...,5.0,330,True,0,0,0,0,0,1,0
8,9/22/2026 21:33:41,2026-09-22,01:00:00,02:33:00,Writing,4,1,20.0,3.0,20,...,2.0,93,False,0,0,0,0,0,1,0
9,9/23/2026 8:29:03,2026-09-23,09:00:00,10:00:00,Reading,8,4,120.0,15.0,5,...,2.0,60,False,0,0,0,0,0,1,0


## 11. Handling missing values

After cleaning, few columns still have missing values.

**Why not fill with 0?** A 0 here would claim the person slept 0 hours or
used their phone for 0 minutes, which is a real (and very different)
value. It would drag averages down and distort the sleep vs. focus
comparison. Instead we:

1. Fill numeric gaps with the **median** of that column, which is a
   typical value and isn't pulled around by outliers like a 600-minute
   screen time.
2. Add an `_imputed` flag column (`True` where we filled a value), so any
   chart or analysis can leave those rows out. For the sleep vs. focus
   question in particular, we recommend filtering to
   `sleep_hours_imputed == False` rather than plotting a made-up point.
3. Fill any missing **categorical** column with `"Unknown"`. Right now no
   categorical column has gaps (the literal `"None"` answers in
   `caffeine` and `interaction_level` are real answers, not missing
   data), but this keeps the notebook safe if more responses come in.

Columns where a 0 is a meaningful answer are deliberately left alone, for
example `time_to_first_distraction_min = 0` means "no distraction
occurred" and `num_distractions = 0` means none happened.

In [18]:
# add planned_duration_min to the list of columns that get median-filled + flagged
NUMERIC_TO_FILL = ["Sleep_Hours", "Phone_Screen_Time", "Planned_Duration"]


for col in NUMERIC_TO_FILL:
    flag_col = f"{col}_imputed"
    df[flag_col] = df[col].isna()
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"{col}: filled {df[flag_col].sum()} missing value(s) with median = {median_value}")

CATEGORICAL_COLS = ['Task_Type', "Location", "Noise_Level", "People_Present",
                    "Interaction_Level", "Caffeine"]

for col in CATEGORICAL_COLS:
    n_missing = df[col].isna().sum()
    if n_missing:
        df[col] = df[col].fillna("Unknown")
        print(f"{col}: filled {n_missing} missing value(s) with 'Unknown'")

print()
print("Missing values remaining:", int(df.isna().sum().sum()))

Sleep_Hours: filled 1 missing value(s) with median = 6.0
Phone_Screen_Time: filled 3 missing value(s) with median = 180.0
Planned_Duration: filled 0 missing value(s) with median = 90.0

Missing values remaining: 0


## 12. Save the Cleaned Data

In [19]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved {len(df)} rows x {len(df.columns)} columns to {PROCESSED_PATH}")

Saved 22 rows x 32 columns to /content/drive/MyDrive/cleaned_data.csv
